In [3]:
#| hide
%load_ext autoreload
%autoreload 2

# Search

> Functionalities to search, and retrieve data from pubmed

In [1]:
#| default_exp PubmedSearch

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| export
from Bio import Entrez
import os
from datetime import datetime, timedelta, date
from fastcore.all import *
from typing import Union, Optional, Any
from pydantic import BaseModel, ValidationError, model_validator, field_validator


In [7]:
#| export
from pubmed_lib.data import *
from pubmed_lib.result import *
from pubmed_lib.parser import *

In [8]:
#| hide
from dotenv import load_dotenv, find_dotenv

In [9]:
#| hide
load_dotenv(find_dotenv())
email = os.environ.get('EMAIL')
api_key = os.environ.get('ENTREZ_API_KEY')

In [10]:
#| exports

class Search(BaseModel):
    """
    Search class to warp the search and results
    """
    search_tag:str  = 'Title/Abstract' #Tag to specifiy the search, can be any from pubmed, Defaul: Title/Abstract
    retmax:int = 200 #Maximum number of results to be retrieved
    retmode:str ='xml' #Format of the returned data, options are xml, 
    sort:str='relevance' #Way to sort the results
    mindate: int | None = None #Initial data to be search from, year
    maxdate: int | None = None #Final data to be search from, year
    idlist: List[int] | None = None
    email:str | None = None
    api_key:str | None = None
    
    @model_validator(mode='before')
    def validate_email(cls,values:dict )->dict:
        email = get_from_dict_or_env(
            values, "email", "EMAIL"
        )
        values["email"] = email
        
        api_key = get_from_dict_or_env(values, 'api_key', 'ENTREZ_API_KEY')
        values['api_key'] = api_key
        return values
        
    @field_validator('search_tag', mode='before')
    @classmethod
    def validate_search_tag(cls, v):
        if not v:
            v = 'Title/Abstract'
        if v not in SEARCH_TAGS.keys():
            raise ValueError(f'Search tag need to be some of {SEARCH_TAGS.keys()}')
        return SEARCH_TAGS[v]
    
     

In [11]:
#| exports

@patch
def search(
    self:Search,
    query: str, #Query to be search in pubmed
):
    """
    It receive a query to be searched in pubmed and return the handler of the search
    """
    Entrez.email = self.email
    Entrez.api_key = self.api_key
    query = query+self.search_tag
    handle = Entrez.esearch(db='pubmed',
                    sort=self.sort,
                    retmax=self.retmax,
                    retmode=self.retmode,
                    term=query,
                    mindate = self.mindate,
                    maxdate =self. maxdate)
    results = Entrez.read(handle)
    return results['IdList']

In [12]:
search = Search(search_tag='Author', mindate=2019, maxdate=2025)

In [13]:
search

Search(search_tag='[au]', retmax=200, retmode='xml', sort='relevance', mindate=2019, maxdate=2025, idlist=None, email='dmaturana@ciq.uchile.cl', api_key='7e6d52ff3b753d51c75b38981b98ee477608')

In [14]:
idlist = search.search('Daniel Maturana')

In [15]:
idlist

['33558635']

In [16]:
#| export
@patch
def fetch_details(
    self:Search,
    idlist:List[int], #list of pubmedid to be retreived
    ):
    """
    It receive a list of pubmedIds from a search, and retrieve all the details of those publications
    """
    ids = ','.join(idlist)
    handle = Entrez.efetch(db='pubmed',
                           retmode=self.retmode,
                           id=ids)
    results = Entrez.read(handle)
    return results['PubmedArticle']

In [17]:
#| exports
@patch
def results(
    self:Search,
    query:str, #Term to be queried in pubmed
)->list:
    """
    Method that do the search and retrieve a generator with all the infomration of the articles"""
    results = []
    id_list = self.search(query)
    articles = self.fetch_details(id_list)
    for article in articles:
        article_dict = parse_paperinfo(article)
        results.append( Result.model_validate(article_dict))
    return results


In [20]:
results = search.results('Daniel Maturana')

entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated
entering
error en parsing nor validated


In [22]:
results[0].model_dump()

{'pubmed': '33558635',
 'pmc': 'PMC7870875',
 'doi': '10.1038/s41598-021-82833-w',
 'pii': '10.1038/s41598-021-82833-w',
 'abstract': 'Despite unprecedented global efforts to rapidly develop SARS-CoV-2 treatments, in order to reduce the burden placed on health systems, the situation remains critical. Effective diagnosis, treatment, and prophylactic measures are urgently required to meet global demand: recombinant antibodies fulfill these requirements and have marked clinical potential. Here, we describe the fast-tracked development of an alpaca Nanobody specific for the receptor-binding-domain (RBD) of the SARS-CoV-2 Spike protein with potential therapeutic applicability. We present a rapid method for nanobody isolation that includes an optimized immunization regimen coupled with VHH library E. coli surface display, which allows single-step selection of Nanobodies using a simple density gradient centrifugation of the bacterial library. The selected single and monomeric Nanobody, W25, b

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()